# 07 Faithfulness of the attention explanation (C1 Adaptive Curriculum Engine)

The service explains a recommendation by naming one prerequisite: the argmax over the incoming edges of **mean-head attention x (1 - mastery)**. This notebook asks whether that explanation is *faithful*, that is, whether the edge it names really matters to the model's prediction more than other edges do.

**Test (edge removal).** For every held-out test answer on an exercise with at least 2 prerequisites, run `gat-v2` three more times, each with one prerequisite edge removed (its neighbour slot masked out of the attention):

| Removed edge | Meaning |
|---|---|
| **named** | the edge the explanation names |
| **random** | a different prerequisite edge, chosen at random |
| **lowest** | the edge the explanation ranks last |

The change in predicted P(correct) is |p(full) - p(edge removed)|. A faithful explanation means removing the **named** edge changes the prediction more than removing a random one (and much more than the lowest one). Tests are paired and computed per student (answers of one student are not independent): paired t-test and Wilcoxon signed-rank on each student's mean change.

Only message passing is removed: the target's own features (including `mean_prereq_mastery`) stay as they were, so the test isolates the attention pathway the explanation is read from. A second variant also recomputes `mean_prereq_mastery` and `log_prereq_count` without the edge.

Uses the exported `gat-v2.npz` (no training). Needs notebooks 01 to 04. CPU is enough.

In [1]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 1,000 test students, for a fast check.
QUICK = False

In [2]:
import sys
!{sys.executable} -m pip install -q pyarrow scikit-learn networkx scipy

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "faithfulness"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for needed in ('interactions.parquet', 'splits.json', 'bkt/params.csv', 'graph/edge_index.npy', 'gat/feature_spec.json'):
    assert (PROCESSED / needed).exists(), f"{needed} missing: run the earlier notebooks first."
print("Storage:", STORAGE, "->", BASE)

Mounted at /content/drive
Storage: drive -> /content/drive/MyDrive/adaptlearn-c1


## 1. Load the answers, the graph, the BKT parameters and gat-v2

In [3]:
import hashlib
import json
import math
from collections import deque

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm

SEED = 42
rng = np.random.default_rng(SEED)

inter = pd.read_parquet(PROCESSED / "interactions.parquet",
                        columns=["user_id", "exercise_idx", "step", "correct"])
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())
names = sorted(exercise_index, key=exercise_index.get)
n_ex = len(names)
edge_index = np.load(PROCESSED / "graph" / "edge_index.npy")  # row 0 prerequisite, row 1 exercise
bkt = pd.read_csv(PROCESSED / "bkt" / "params.csv").set_index("exercise").loc[names]

# gat-v2 exactly as exported by notebook 04 (and run by curriculum-service).
spec = json.loads((PROCESSED / "gat" / "feature_spec.json").read_text())
model_path = PROCESSED / "gat" / f"{spec['model_version']}.npz"
model_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()
assert model_sha == spec["sha256"], "gat model file does not match its feature spec"
weights = dict(np.load(model_path))
ARCH = spec["architecture"]
print(f"{spec['model_version']} sha256 {model_sha}")

G = nx.DiGraph()
G.add_nodes_from(range(n_ex))
G.add_edges_from(zip(edge_index[0].tolist(), edge_index[1].tolist()))
assert nx.is_directed_acyclic_graph(G)
depth = np.zeros(n_ex)
for v in nx.topological_sort(G):
    preds = list(G.predecessors(v))
    depth[v] = 1 + max(depth[p] for p in preds) if preds else 0
prereqs = [sorted(G.predecessors(v)) for v in range(n_ex)]
K = max(len(p) for p in prereqs)

gat-v2 sha256 914a3046ac95daa2a58228de2ac6e75811c83f625760c826594123bf2c9cb17d


In [4]:
CONFIG = {"SEED": SEED, "QUICK": QUICK, "MIN_PREREQS": 2}
test_ids = sorted(splits["test"])
if QUICK:
    test_ids = sorted(rng.choice(test_ids, min(1_000, len(test_ids)), replace=False).tolist())
test = inter[inter.user_id.isin(set(test_ids))].reset_index(drop=True)
print(f"{test.user_id.nunique():,} test students, {len(test):,} answers; "
      f"max prerequisites per exercise K = {K}")

6,000 test students, 1,178,267 answers; max prerequisites per exercise K = 7


## 2. Features (answers before *t* only) and the model

In [5]:
# Notebook 04's features, from answers before t only, over each student's whole history.
FEATURES = spec["features"]
LEARNER = spec["learner_features"]
assert FEATURES == ["mastery", "log_evidence", "log_depth", "log_prereq_count",
                    "mean_prereq_mastery", "bkt_p_correct"]
assert LEARNER == ["overall_correct_rate", "recent_correct_rate", "previous_correct"]
BKT_P = FEATURES.index("bkt_p_correct")
F, NL, RECENT = len(FEATURES), len(LEARNER), 5
P_INIT, P_LEARN, P_GUESS, P_SLIP = (bkt[c].to_numpy().tolist()
                                    for c in ("p_init", "p_learn", "p_guess", "p_slip"))
LOG_DEPTH = np.log1p(depth).tolist()
LOG_PREREQS = [math.log1p(len(p)) for p in prereqs]


def node_features(v, state, count):
    ps = prereqs[v]
    mean_pm = sum(state[p] for p in ps) / len(ps) if ps else 1.0
    m = state[v]
    return (m, math.log1p(count[v]), LOG_DEPTH[v], LOG_PREREQS[v], mean_pm,
            m * (1 - P_SLIP[v]) + (1 - m) * P_GUESS[v])


def build_features(frame):
    """X_t [n,F], X_n [n,K,F], M [n,K], X_l [n,NL] for every row of `frame` (sorted by user, step)."""
    n = len(frame)
    X_t = np.zeros((n, F), np.float32)
    X_n = np.zeros((n, K, F), np.float32)
    M = np.zeros((n, K), bool)
    X_l = np.zeros((n, NL), np.float32)
    users = frame.user_id.to_numpy()
    ex = frame.exercise_idx.to_numpy().tolist()
    y = frame.correct.to_numpy().astype(bool).tolist()
    new_student = np.r_[True, users[1:] != users[:-1]].tolist()
    for i in tqdm(range(n), mininterval=5):
        if new_student[i]:
            state, count = list(P_INIT), [0] * n_ex
            answered, right, recent, previous = 0, 0, deque(maxlen=RECENT), 0.5
        X_l[i] = ((right + 1) / (answered + 2), (sum(recent) + 1) / (len(recent) + 2), previous)
        v = ex[i]
        X_t[i] = node_features(v, state, count)
        for j, u in enumerate(prereqs[v]):
            X_n[i, j] = node_features(u, state, count)
            M[i, j] = True
        m, g, s = state[v], P_GUESS[v], P_SLIP[v]
        if y[i]:
            post = m * (1 - s) / (m * (1 - s) + (1 - m) * g)
        else:
            post = m * s / (m * s + (1 - m) * (1 - g))
        state[v] = post + (1 - post) * P_LEARN[v]
        count[v] += 1
        answered += 1
        right += y[i]
        recent.append(y[i])
        previous = float(y[i])
    return X_t, X_n, M, X_l


def gat_forward_numpy(w, x_t, x_n, mask, x_l, heads, hidden, slope):
    """Notebook 04's reference forward pass (also copied into curriculum-service)."""
    B, k, _ = x_n.shape
    nodes = np.concatenate([x_t[:, None], x_n], 1)
    h = (nodes @ w["W"]).reshape(B, 1 + k, heads, hidden)
    score = (h[:, :1] * w["a_dst"]).sum(-1) + (h * w["a_src"]).sum(-1)
    score = np.where(score > 0, score, slope * score)
    present = np.concatenate([np.ones((B, 1), bool), mask], 1)
    score = np.where(present[..., None], score, -np.inf)
    score = score - score.max(1, keepdims=True)
    alpha = np.exp(score) / np.exp(score).sum(1, keepdims=True)
    z = (alpha[..., None] * h).sum(1).reshape(B, heads * hidden)
    z = np.where(z > 0, z, np.expm1(np.minimum(z, 0)))
    hidden1 = np.maximum(np.concatenate([z, x_t, x_l], -1) @ w["fc1_w"] + w["fc1_b"], 0)
    p_bkt = np.clip(x_t[:, BKT_P], 1e-4, 1 - 1e-4)
    logit = np.log(p_bkt / (1 - p_bkt)) + (hidden1 @ w["fc2_w"] + w["fc2_b"])[:, 0]
    return 1 / (1 + np.exp(-logit)), alpha


def gat(x_t, x_n, mask, x_l, batch=200_000):
    out_p, out_a = [], []
    for b in range(0, len(x_t), batch):
        s = slice(b, b + batch)
        p, a = gat_forward_numpy(weights, x_t[s], x_n[s], mask[s], x_l[s],
                                 ARCH["heads"], ARCH["hidden"], ARCH["leaky_slope"])
        out_p.append(p)
        out_a.append(a)
    return np.concatenate(out_p), np.concatenate(out_a)

In [6]:
from sklearn.metrics import roc_auc_score

X_t, X_n, M, X_l = build_features(test)
y = test.correct.to_numpy().astype(bool)
p_full, alpha = gat(X_t, X_n, M, X_l)
auc = roc_auc_score(y, p_full)
print(f"gat-v2 on every test answer: AUC {auc:.4f} (notebook 04 reported "
      f"{next(r['auc'] for r in spec['test_results'] if r['model'].startswith('BKT + GAT')):.4f})")
if not QUICK:
    assert abs(auc - next(r["auc"] for r in spec["test_results"] if r["model"].startswith("BKT + GAT"))) < 0.002, \
        "features differ from notebook 04"

  0%|          | 0/1178267 [00:00<?, ?it/s]

gat-v2 on every test answer: AUC 0.7930 (notebook 04 reported 0.7930)


## 3. Remove one edge at a time

`weakness = attention x (1 - mastery)` per present edge (attention averaged over heads, slot 0 is the self-loop and is never removed). The named edge is its argmax, the lowest edge its argmin, the random edge any other present edge.

In [7]:
rows = np.flatnonzero(M.sum(1) >= CONFIG["MIN_PREREQS"])
xt, xn, m, xl, p0 = X_t[rows], X_n[rows], M[rows], X_l[rows], p_full[rows]
edge_att = alpha[rows, 1:].mean(-1)                     # [B, K]
weakness = np.where(m, edge_att * (1 - xn[..., 0]), np.nan)
named = np.nanargmax(weakness, 1)
lowest = np.nanargmin(weakness, 1)
pick_rng = np.random.default_rng(SEED)
random_edge = np.array([pick_rng.choice([j for j in np.flatnonzero(mm) if j != t])
                        for mm, t in zip(m, named)])


def without(slot, recompute_features=False):
    mask = m.copy()
    mask[np.arange(len(mask)), slot] = False
    x_t = xt.copy()
    if recompute_features:  # variant: the target's graph features also lose the edge
        kept = mask.sum(1)
        x_t[:, FEATURES.index("mean_prereq_mastery")] = np.where(
            kept > 0, (xn[..., 0] * mask).sum(1) / np.maximum(kept, 1), 1.0)
        x_t[:, FEATURES.index("log_prereq_count")] = np.log1p(kept)
    return gat(x_t, xn, mask, xl)[0]


change, flipped = {}, {}
for variant, recompute in (("mask_only", False), ("with_features", True)):
    for name, slot in (("named", named), ("random", random_edge), ("lowest", lowest)):
        p1 = without(slot, recompute)
        change[(variant, name)] = np.abs(p0 - p1)
        flipped[(variant, name)] = (p0 >= 0.5) != (p1 >= 0.5)
print(f"{len(rows):,} answers on exercises with >= {CONFIG['MIN_PREREQS']} prerequisites "
      f"({len(rows) / len(y):.1%} of test answers), "
      f"{pd.Series(test.user_id.to_numpy()[rows]).nunique():,} students")

387,456 answers on exercises with >= 2 prerequisites (32.9% of test answers), 4,850 students


## 4. Does removing the named edge matter more?

In [8]:
from scipy.stats import ttest_rel, wilcoxon

owner = test.user_id.to_numpy()[rows]
summary_rows, tests = [], {}
for variant in ("mask_only", "with_features"):
    d = {name: change[(variant, name)] for name in ("named", "random", "lowest")}
    per_student = pd.DataFrame({**d, "owner": owner}).groupby("owner").mean()
    for name, values in d.items():
        summary_rows.append({"variant": variant, "removed_edge": name,
                             "mean_change": values.mean(), "median_change": np.median(values),
                             "flips_decision_at_0.5": float(flipped[(variant, name)].mean())})
    for other in ("random", "lowest"):
        t, p_t = ttest_rel(per_student["named"], per_student[other])
        w, p_w = wilcoxon(per_student["named"], per_student[other])
        tests[f"{variant}: named vs {other}"] = {
            "students": len(per_student),
            "named_larger_share_of_answers": float((d["named"] > d[other]).mean()),
            "mean_difference": float((per_student["named"] - per_student[other]).mean()),
            "paired_t": float(t), "p_t": float(p_t), "wilcoxon": float(w), "p_wilcoxon": float(p_w)}
summary = pd.DataFrame(summary_rows)
display(summary.round(4))
display(pd.DataFrame(tests).T.round(4))

,variant,removed_edge,mean_change,median_change,flips_decision_at_0.5
0,mask_only,named,0.0060,0.0029,0.0047
1,mask_only,random,0.0070,0.0034,0.0056
2,mask_only,lowest,0.0071,0.0034,0.0058
3,with_features,named,0.0061,0.0034,0.0047
4,with_features,random,0.0072,0.0043,0.0055
5,with_features,lowest,0.0074,0.0043,0.0057


,students,named_larger_share_of_answers,mean_difference,paired_t,p_t,wilcoxon,p_wilcoxon
mask_only: named vs random,4850.0,0.4596,0.0000,0.1314,0.8955,4947893.0,0.0000
mask_only: named vs lowest,4850.0,0.4438,-0.0001,-0.9350,0.3498,4802242.0,0.0000
with_features: named vs random,4850.0,0.4125,0.0007,6.3753,0.0000,5646597.0,0.0159
with_features: named vs lowest,4850.0,0.3973,0.0005,4.3602,0.0000,5375901.0,0.0000


In [9]:
# By number of prerequisites (mask_only): more edges = a harder choice for the explanation.
n_prereq = m.sum(1)
by_k = pd.DataFrame({"prerequisites": n_prereq, "named": change[("mask_only", "named")],
                     "random": change[("mask_only", "random")]})
by_k["named_larger"] = by_k["named"] > by_k["random"]
display(by_k.groupby("prerequisites").agg(answers=("named", "size"), named=("named", "mean"),
                                           random=("random", "mean"),
                                           named_larger=("named_larger", "mean")).round(4))

,answers,named,random,named_larger
prerequisites,,,,
2,288368,0.0063,0.0077,0.4487
3,61852,0.0066,0.0061,0.5194
4,18292,0.0044,0.0040,0.5030
5,5542,0.0024,0.0027,0.4917
6,12900,0.0007,0.0014,0.3346
7,502,0.0041,0.0033,0.6175


## 5. Save results and a manifest

In [10]:
summary.to_csv(OUT_DIR / "edge_removal.csv", index=False)
_ = (OUT_DIR / "tests.json").write_text(json.dumps({
    "answers": int(len(rows)), "share_of_test_answers": float(len(rows) / len(y)),
    "gat_test_auc_all_answers": float(auc), "tests": tests,
}, indent=2))

In [11]:
import platform, scipy, sklearn
from datetime import datetime, timezone

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "model": {"version": spec["model_version"], "sha256": model_sha},
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "scipy": scipy.__version__,
                 "scikit-learn": sklearn.__version__, "networkx": nx.__version__},
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive(f"/content/{OUT_DIR.name}", "zip", OUT_DIR))

{
  "created_at": "2026-10-11T14:57:40.283284+00:00",
  "config": {
    "SEED": 42,
    "QUICK": false,
    "MIN_PREREQS": 2
  },
  "model": {
    "version": "gat-v2",
    "sha256": "914a3046ac95daa2a58228de2ac6e75811c83f625760c826594123bf2c9cb17d"
  },
  "versions": {
    "python": "3.13.15",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "networkx": "3.7"
  },
  "sha256": {
    "edge_removal.csv": "125d95cec1bf52195663c12c5a49786045a0f0c6e7f46c4058e4cdff9e2501c8",
    "tests.json": "5bf18440c80038a3e95f7b21fea49a1228b904d0619af5a0494107735703daf7"
  }
}


## For the report

Report the edge-removal table (mean change in P(correct) when the named, a random and the lowest-ranked edge is removed) and the paired tests per student.

- **Faithful:** named > random with p < 0.05 (both variants) and named > lowest. Then the sentence the learner sees names an edge the model actually relies on.
- **Not faithful:** named is not larger than random. Report it as a limitation: notebook 06 already found that attention adds no accuracy (no_attention -0.0001, n.s.), so the explanation would then rest on mastery (1 - mastery picks the edge) rather than on what the model attends to. The service's sentence stays truthful either way, because it names a prerequisite the learner has not mastered.

Method sentence: *Explanation faithfulness was measured by edge removal (cf. DeYoung et al., 2020, comprehensiveness): for each held-out answer with two or more prerequisites, the prediction change from masking the named edge was compared with masking a random and the lowest-ranked edge, using paired tests on per-student means.*

Before committing: **Clear all outputs**.